In [1]:
import streamlit as st
from streamlit_jupyter import StreamlitPatcher, tqdm
StreamlitPatcher().jupyter()  # register streamlit with jupyter-compatible wrappers
import sys; sys.path.append('..')
from osp import *
pd.options.display.max_colwidth = 200
pd.options.display.max_rows = 20

In [2]:
@cache
def get_current_pred_probs(target_col='discipline'):
    df_preds = get_df_preds()
    num_runs = df_preds['run'].nunique()
    odf = df_preds.groupby(['predict_type','comparison','id']).mean(numeric_only=True).reset_index().drop(columns=['run'])
    odf['target'] = odf['id'].map(lambda x: get_text_metadata(x).get(target_col,''))
    return odf.sort_values('prob_Philosophy',ascending=False).set_index('id').assign(num_runs=num_runs)

In [3]:
df_preds = get_current_pred_probs()
df_feats = get_current_feat_weights(group_by=('feature','comparison'))

In [4]:
df_preds

,predict_type,comparison,prob_Literature,prob_Philosophy,support,target,num_runs
id,,,,,,,
phil/10.2307/48692538__03,cv,2000-2025 Philosophy vs 2000-2025 Literature,0.000000e+00,1.000000e+00,2000.0,Philosophy,10
phil/10.2307/48692275__03,cv,2000-2025 Philosophy vs 2000-2025 Literature,1.820766e-14,1.000000e+00,2000.0,Philosophy,10
phil/10.2307/20118078__02,cv,1975-2000 Philosophy vs 1975-2000 Literature,5.773160e-14,1.000000e+00,2000.0,Philosophy,10
phil/10.2307/41477596__04,unseen,2000-2025 Philosophy vs 2000-2025 Literature,1.914659e-13,1.000000e+00,2000.0,Philosophy,10
phil/10.2307/24019863__02,cv,2000-2025 Philosophy vs 2000-2025 Literature,1.240785e-12,1.000000e+00,2000.0,Philosophy,10
...,...,...,...,...,...,...,...
phil/10.2307/42964196__01,unseen,1900-1925 Philosophy vs 1900-1925 Literature,1.000000e+00,5.629318e-13,1504.0,Philosophy,10
phil/10.2307/20010421__01,unseen,1900-1925 Philosophy vs 1900-1925 Literature,1.000000e+00,4.811954e-13,1504.0,Philosophy,10
phil/10.2307/41477594__03,unseen,1925-1950 Philosophy vs 1925-1950 Literature,1.000000e+00,3.716023e-15,2000.0,Philosophy,10


In [5]:
TARGET_NICKNAMES = {'Philosophy':'Phil', 'Literature':'Lit'}

def get_slice_info_df_preds(slice_ids):
    if isinstance(slice_ids, str): slice_ids = [slice_ids]
    df_all_preds = get_current_pred_probs()
    df_all_feats = get_current_feat_weights(group_by=('feature','comparison'))

    df_preds = df_all_preds.loc[slice_ids]

    targets = df_preds['target'].unique()
    comparisons = df_preds['comparison'].unique()
    predict_types = df_preds['predict_type'].unique()
    
    def describe_probs_target(target, dfx):
        probf = f'prob_{target}'
        out_d = {}
        out_d2 = {}
        out_d3 = {}
        o = []

        avg_prob = dfx[probf].mean()
        num_correct = len(dfx.query(f'{probf}>=0.5'))
        out_d['prob'] = avg_prob
        # out_d['perc_correct'] = num_correct / len(dfx)

        tname = TARGET_NICKNAMES.get(target, target)
        for cmp in sorted(comparisons):
            cmpname = cmp.split('-')[0]
            dfx_cmp = dfx[dfx['comparison']==cmp]
            avg_cmp_prob = dfx_cmp[probf].mean()
            num_cmp_correct = len(dfx_cmp.query(f'{probf}>=0.5'))
            out_d[f'prob_{cmpname}'] = avg_cmp_prob
            # out_d2[f'perc_correct_{cmpname}'] = num_cmp_correct / len(dfx_cmp)
            for pt in predict_types:
                dfx_cmp_pt = dfx_cmp[dfx_cmp['predict_type']==pt]
                support = dfx_cmp_pt.iloc[0]['support']
                num_runs = dfx_cmp_pt.iloc[0]['num_runs']
                avg_cmp_pt_prob = dfx_cmp_pt[probf].mean()
                num_cmp_pt_correct = len(dfx_cmp_pt.query(f'{probf}>=0.5'))
                perc_cmp_pt_correct = num_cmp_pt_correct / len(dfx_cmp_pt)
                out_d3[f'prob_{cmpname}_{pt}'] = avg_cmp_pt_prob
                outx = {
                    'target':target,
                    'comparison':cmp,
                    'predict_type':pt,
                    'prob_correct':avg_cmp_pt_prob,
                    'perc_correct':perc_cmp_pt_correct,
                    'num_correct':num_cmp_pt_correct,
                    'num_runs':num_runs,
                    'support':support,
                    'num_samples':len(dfx_cmp_pt),
                }
                o.append(outx)
        return o

    ld = []
    for target,target_df in df_preds.groupby('target'):
        out_l = describe_probs_target(target, target_df)
        ld.extend(out_l)
    return pd.DataFrame(ld).set_index('target')


In [6]:
df_preds_sample = df_preds.groupby('target').sample(100)
get_slice_info_df_preds(df_preds_sample.index.tolist())

,comparison,predict_type,prob_correct,perc_correct,num_correct,num_runs,support,num_samples
target,,,,,,,,
Literature,1900-1925 Philosophy vs 1900-1925 Literature,unseen,0.759641,0.793478,73,10,1504.0,92
Literature,1900-1925 Philosophy vs 1900-1925 Literature,cv,0.904182,1.000000,8,10,1504.0,8
Literature,1925-1950 Philosophy vs 1925-1950 Literature,unseen,0.770223,0.777778,77,10,2000.0,99
Literature,1925-1950 Philosophy vs 1925-1950 Literature,cv,0.890336,0.923077,12,10,2000.0,13
Literature,1950-1975 Philosophy vs 1950-1975 Literature,unseen,0.862421,0.910000,91,10,2000.0,100
Literature,1950-1975 Philosophy vs 1950-1975 Literature,cv,0.888227,0.894737,17,10,2000.0,19
Literature,1975-2000 Philosophy vs 1975-2000 Literature,unseen,0.907610,0.970000,97,10,2000.0,100
Literature,1975-2000 Philosophy vs 1975-2000 Literature,cv,0.915331,0.971429,34,10,2000.0,35
Literature,2000-2025 Philosophy vs 2000-2025 Literature,unseen,0.942614,0.960000,96,10,2000.0,100


In [71]:


def describe_slice_probs(slice_ids, width=90, para='\n\n'):
    import textwrap
    out=[]
    dfx = get_slice_info_df_preds(slice_ids)
    dfx_q = dfx.select_dtypes(include=['number'])
    median_cols = ['support','num_correct']
    sum_cols = ['num_samples']
    avg_cols = [c for c in dfx_q.columns if c not in set(median_cols+sum_cols)]
    dfx_target = dfx.groupby('target').agg(
        {
            **{c:'mean' for c in avg_cols},
            **{c:'sum' for c in sum_cols},
            **{c:'median' for c in median_cols},
        }
    )
    
    # target_cols = prob_correct	perc_correct	num_correct	support	num_samples

    num_cmps = dfx.comparison.nunique()
    for target,row in dfx_target.iterrows():
        out.append(f'''- **{target}** (n={int(row.num_samples):,}) was predicted successfully **{row.perc_correct*100:.1f}%** of the time (across {int(row.num_runs):,} model runs each of {num_cmps} comparisons), with an average confidence of **{row.prob_correct*100:.1f}%**. ''')
    
    for target,row in dfx_target.iterrows():
        target_comparison_df = dfx.query('target==@target')
        target_comparison_df.sort_values(['prob_correct','perc_correct'],ascending=False,inplace=True)
        best_cmp = target_comparison_df.iloc[0]
        worst_cmp = target_comparison_df.iloc[-1]
        support = int(best_cmp.support)
        out.append(f'''- Each run of the model had **{support:,}** samples divided evenly.''')
        out.append(f'''- The best performing comparison was **{best_cmp.comparison.split(" ")[0]}** ({best_cmp.prob_correct*100:.1f}% confidence),  with a success rate of **{best_cmp.perc_correct*100:.1f}%**.''')
        out.append(f'''- The worst performing comparison was **{worst_cmp.comparison.split(" ")[0]}** ({worst_cmp.prob_correct*100:.1f}% confidence), with a success rate of **{worst_cmp.perc_correct*100:.1f}%**.''' )
        out
        break

    return para.join(textwrap.fill(x, width=width) for x in out)


In [72]:
dfx=df_preds.query('comparison >= "2000"')
# dfx = df_preds.query('target == "Philosophy"')
# dfx

In [73]:
printm(describe_slice_probs(dfx.index.tolist()))

- **Literature** (n=84,420) was predicted successfully **87.0%** of the time (across 10
model runs each of 5 comparisons), with an average confidence of **83.8%**.

- **Philosophy** (n=168,772) was predicted successfully **90.2%** of the time (across 10
model runs each of 5 comparisons), with an average confidence of **86.8%**.

- Each run of the model had **2,000** samples divided evenly.

- The best performing comparison was **2000-2025** (91.8% confidence),  with a success
rate of **94.9%**.

- The worst performing comparison was **1925-1950** (67.1% confidence), with a success
rate of **68.5%**.

In [10]:
get_slice_info_df_preds(dfx.index.tolist())

,comparison,predict_type,prob_correct,perc_correct,num_correct,num_runs,support,num_samples
target,,,,,,,,
Literature,1900-1925 Philosophy vs 1900-1925 Literature,cv,0.857448,0.888298,668,10,1504.0,752
Literature,1900-1925 Philosophy vs 1900-1925 Literature,unseen,0.719473,0.741998,10014,10,1504.0,13496
Literature,1925-1950 Philosophy vs 1925-1950 Literature,cv,0.888963,0.914643,1243,10,2000.0,1359
Literature,1925-1950 Philosophy vs 1925-1950 Literature,unseen,0.670708,0.685483,9727,10,2000.0,14190
Literature,1950-1975 Philosophy vs 1950-1975 Literature,cv,0.868743,0.905020,2001,10,2000.0,2211
Literature,1950-1975 Philosophy vs 1950-1975 Literature,unseen,0.816842,0.859530,12244,10,2000.0,14245
Literature,1975-2000 Philosophy vs 1975-2000 Literature,cv,0.871858,0.906886,3253,10,2000.0,3587
Literature,1975-2000 Philosophy vs 1975-2000 Literature,unseen,0.854363,0.902934,12865,10,2000.0,14248
Literature,2000-2025 Philosophy vs 2000-2025 Literature,cv,0.918025,0.949047,5774,10,2000.0,6084


In [11]:

def get_slice_info(slice_id, df_preds=None):
    out_d = {}
    if df_preds is None:
        df_preds = get_df_preds()
    
    return df_preds


In [12]:
for id,x in STASH_SLICES_NLP.items():
    break

In [13]:
get_slice_info(id)

,prob_Literature,prob_Philosophy,support,run,predict_type,comparison
id,,,,,,
lit/456903__06,0.021961,0.978039,1504,0,cv,1900-1925 Philosophy vs 1900-1925 Literature
lit/3713754__02,0.977459,0.022541,1504,0,cv,1900-1925 Philosophy vs 1900-1925 Literature
lit/457155__03,0.996810,0.003190,1504,0,cv,1900-1925 Philosophy vs 1900-1925 Literature
lit/433422__05,0.991444,0.008556,1504,0,cv,1900-1925 Philosophy vs 1900-1925 Literature
lit/456699__05,0.161604,0.838396,1504,0,cv,1900-1925 Philosophy vs 1900-1925 Literature
...,...,...,...,...,...,...
lit/27760277__03,0.979312,0.020688,2000,9,unseen,2000-2025 Philosophy vs 2000-2025 Literature
phil/10.2307/2025409__04,0.000012,0.999988,2000,9,unseen,2000-2025 Philosophy vs 2000-2025 Literature
phil/10.2307/2953730__03,0.010580,0.989420,2000,9,unseen,2000-2025 Philosophy vs 2000-2025 Literature


In [14]:
display_slice_predictions(doc, 'weight_z', 'word')

NameError: name 'display_slice_predictions' is not defined